In [1]:
import pandas as pd
import pymysql
import unicodedata
import re
from nltk.corpus import stopwords
import string
from collections import Counter

In [5]:
conexion = pymysql.connect(
    host="dl-radar.cluster-ro-c7pmwdslewrp.us-east-1.rds.amazonaws.com",
    user = "debian",
    password= "eeAZU3v1FXCY9zmbvcS6kpEpyj",
    database="data_fact",
    port= 4408
)

cursor = conexion.cursor()

In [6]:
query = """
SELECT *
FROM base_rucs_sri;
"""
base_registro_civil = pd.read_sql_query(query, conexion)

C:\Users\anali\AppData\Local\Temp\ipykernel_29500\631102470.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  base_registro_civil = pd.read_sql_query(query, conexion)


In [498]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\bases\base_rucs_sri.parquet")

In [134]:
def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def limpiador(lista: list):
    documentos = []
    re_punctuation = re.compile('[%s]' % re.escape(string.punctuation))

    stop_words_s = set(stopwords.words('spanish'))
    stop_words = set(stopwords.words('english'))

    for descripcion in lista:
        tokens = descripcion.split()
        tokens = [re_punctuation.sub(' ', w) for w in tokens]
        tokens = ' '.join(tokens).split()

        tokens = [quitar_tildes(word.lower()) for word in tokens]

        tokens = [word.lower() for word in tokens if re.search('[a-z ]', word.lower())]
        tokens = [word.lower() for word in tokens if word.isalpha()]
        
        tokens = [w for w in tokens if w not in stop_words_s]
        tokens = [w for w in tokens if w not in stop_words]
        
        tokens = [word for word in tokens if len(word) > 2]

        if len(tokens) > 0:
            documento = ' '.join(tokens)
        else:
            documento = ''  # ← asegura longitud igual al DF

        documentos.append(documento)

    return documentos

In [135]:
base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)


In [136]:
base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()]

In [137]:
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))

In [138]:
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [139]:
recreo_nombres = pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\tablas_locales_CC\ccrecreo.xlsx")

In [ ]:
recreo_nombres= pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\tablas_locales_CC\Tabla maestro Condado 3 3.xlsx")

In [142]:
recreo_nombres.columns

Index(['num', 'nombre_establecimiento', 'RUC', 'CÓDIGO ESTABLECIMIENTO',
       'ESTADO', 'CÓDIGO ESTABLECIMIENTO.1', 'id_establecimiento', 'categoria',
       'tipo_local', 'posible'],
      dtype='object')

In [141]:
len(recreo_nombres)

198

In [143]:
dicc_nombre_fantasia = set(recreo_nombres['nombre_establecimiento'])

In [144]:
#Normalizamos los q gramas
def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [145]:
#parte una palabra en la cantidad de q gramas dados
def qgrams(s, q=3):
    return {s[i:i+q] for i in range(len(s) - q + 1)}

In [146]:
#Es la medida que vamos a tomar. La cantidad de q gramas dados los que coinciden dividido para todos
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [305]:
# Normalizamos los nombres fantasia
dic_norm = {normalizar(x): x for x in dicc_nombre_fantasia}

dic_qgrams = {
    k: qgrams(k, q=3)
    for k in dic_norm.keys()
}

In [432]:
#Match de los qgramas en los centros comerciales
def match_qgram(nombre,centro_comercial, threshold=0.5, umbral_centro_comercial = 0.28):
    nombre_sin_centro_comercial = nombre.lower().replace(centro_comercial, "").strip()
    s = normalizar(nombre_sin_centro_comercial)
    q_s = qgrams(s)

    mejor, score = None, 0
    for k, q_k in dic_qgrams.items():
        sim = jaccard(q_s, q_k)
        if sim > score:
            mejor, score = dic_norm[k], sim

    if score >= threshold:
        return mejor, score
    elif (score>= umbral_centro_comercial) and (centro_comercial.lower() in nombre.lower()):
        return mejor , 1 
    return None, score

In [504]:
#Normalizamos el nombre fantasía comercial
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar)

#Tenemos la direccion completa separada en base registro civil
base_registro_civil[['provincia',  'canton', 'parroquia', 'calles']] = (
    base_registro_civil['direccion_completa']
        .str.split('/', n=3, expand=True)
)

In [434]:
#Para sacarnos las frecuencias

#Nombre del centro comercial
nombre = 'san'
apellido = 'marino'
rejex = f'(?=.*{nombre})(?=.*{apellido})'

#Filtros para las calles
mask_regex = base_registro_civil['nombre_fantasia_comercial'].str.contains(rf'{rejex}', case = False, na = False)
#mask_canton = base_registro_civil['canton'].str.contains("Quito", case = False, na =  False)
#mask_parroquia =  base_registro_civil['parroquia'].str.contains("magda", case = False, na =  False)

#Filtramos la base para obtener las calles
base_filtrada = base_registro_civil[mask_regex]

#Obtenemos las calles
list_calles =list(base_filtrada['calles'])

#Limpiamos las calles y hacemos una sola list
list_calles_limpia = limpiador(list_calles)
list_calles_limpia_total = [
    palabra
    for i in range(len(list_calles_limpia))
    for palabra in list_calles_limpia[i].split()
]

#Sacamos las frecuencias
frecuencias = Counter(list_calles_limpia_total)

#Hacemos dataframe de frecuencias
df_frecuencias = ( 
    pd.DataFrame(frecuencias.items(), columns = ['palabra', 'frecuencia'])
    .sort_values('frecuencia', ascending = False)
    .reset_index(drop = True)
)


In [435]:

#Guardamos las frecuencias
#df_frecuencias.to_excel(rf"frecuencias_{nombre}_{apellido}.xlsx", index = False)

In [436]:
lista_palabras_acotacion = ["antonio", "jose", "sucre", "prensa", "john", "kennedy", "jose", "soto", "Leonardo", "davinci", "mariscal", "sucre", "nogales"]

lista_acotacion_especifica = lista_palabras_acotacion +['av']

str_clave = "|".join(lista_palabras_acotacion)

In [508]:
mask_canton = base_registro_civil['canton'].str.contains("QUITO", case = False, na = False) 
mask_parroquia = base_registro_civil['parroquia'].str.contains("ponceano|cotocollao", case = False, na = False)
#mask_calle_mald  = base_registro_civil['calles'].str.contains("", case = False, na = False)
mask_calles = base_registro_civil['calles'].str.contains(f"{str_clave}", case = False, na = False)

mask_general = mask_canton & (mask_parroquia & mask_calles)

In [509]:
base_registro_civil_test = base_registro_civil[mask_general].copy()

In [535]:
base_registro_civil_test[['matcheo_qgram', 'score_qgram']] = (
    base_registro_civil_test['nombre_fantasia_comercial']
      .apply(lambda x: pd.Series(match_qgram(x, centro_comercial='condado', threshold=0.45, umbral_centro_comercial=0.5)))
)

In [536]:
def porcentaje_coincidencias(calle:str):
    num_coincidencia = 0
    join_acotacion_especifica = " ".join(lista_acotacion_especifica).lower()
    split_calle = re.split(r"[.\s]+", calle.lower())
    for palabra in split_calle:
        if palabra in join_acotacion_especifica:
            num_coincidencia =num_coincidencia + 1
    return num_coincidencia/len(split_calle)

In [537]:
# Cantidad porcentaje match que hay
total_palabras = len(lista_acotacion_especifica)
base_registro_civil_test['porcentaje_match'] = (base_registro_civil_test['calles'].apply(porcentaje_coincidencias))

In [538]:
# Normalizacion [0,2] y score general
base_registro_civil_test["score_qgram_02"] =2*base_registro_civil_test['score_qgram']
base_registro_civil_test["porcentaje_match_02"] =2*base_registro_civil_test['porcentaje_match']
#Score General
base_registro_civil_test['score_general'] = base_registro_civil_test["porcentaje_match_02"]*base_registro_civil_test["score_qgram_02"]

In [539]:
base_registro_civil_test = base_registro_civil_test.sort_values(by = "score_general") 

In [540]:
base_registro_civil_test[base_registro_civil_test['parroquia'].str.contains('cotocollao', case = False, na = False)]

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,provincia,canton,parroquia,calles,matcheo_qgram,score_qgram,porcentaje_match,score_qgram_02,porcentaje_match_02,score_general
28464,1.009876e+14,1.009876e+11,1.0,SANCHEZ GAVILANEZ JOSÉ GILBERTO,,3.0,SUSPENDIDO,2.0,CERRADO,1,...,PICHINCHA,QUITO,COTOCOLLAO,CARLOS QUINTO Y JOSE MANOSALVAS,NaN,0.0,0.500000,0.0,1.000000,0.000000
7929581,2.450278e+15,2.450278e+12,1.0,GONZALEZ CASTRO GENESIS FIORELLA,,1.0,ACTIVO,1.0,ABIERTO,1,...,PICHINCHA,QUITO,COTOCOLLAO,LA ESPERANZA N71-700 Y AV. MARISCAL SUCRE,NaN,0.0,0.625000,0.0,1.250000,0.000000
7922136,2.400157e+15,2.400157e+12,1.0,GONZABAY MUÑOZ LISSETTE ELIZABETH,,3.0,SUSPENDIDO,2.0,CERRADO,1,...,PICHINCHA,QUITO,COTOCOLLAO,DE LA PRENSA N64-125 Y N65 BELLAVISTA,NaN,0.0,0.375000,0.0,0.750000,0.000000
7901970,2.350344e+15,2.350344e+12,2.0,SAQUICELA NIVELO VERONICA LILIANA,,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,FCO. PACHECO S/N Y JOSE H. FIGUEROA,NaN,0.0,0.500000,0.0,1.000000,0.000000
7900413,2.350221e+15,2.350221e+12,1.0,ANCHUNDIA ANCHUNDIA MARIA JOSE,,1.0,ACTIVO,1.0,ABIERTO,1,...,PICHINCHA,QUITO,COTOCOLLAO,AV DE LA PRENSA S/N Y DAVID LEDESMA,NaN,0.0,0.444444,0.0,0.888889,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5711962,1.707363e+15,1.707363e+12,20.0,ESTRELLA NARANJO LUCY YELENA,mix two,3.0,SUSPENDIDO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV. MARISCAL SUCRE S/N Y AV. LA PRENSA,MIX TWO,1.0,0.777778,2.0,1.555556,3.111111
6213449,1.714412e+15,1.714412e+12,5.0,RECALDE LOPEZ PABLO SALOMON,mikrotoys,3.0,SUSPENDIDO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,JOHN F. KENNEDY OE4G Y AV. MARISCAL SUCRE,MIKROTOYS,1.0,0.777778,2.0,1.555556,3.111111
7158291,1.790995e+15,1.790995e+12,30.0,ROYALTEX S.A.,lee,1.0,ACTIVO,1.0,ABIERTO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV MARISCAL SUCRE S/N Y JOHN F KENNEDY,LEE,1.0,0.777778,2.0,1.555556,3.111111
166070,1.037832e+14,1.037832e+11,1.0,ORELLANA ESPINOZA ANDREA XIMENA,laiart cuenca,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV JOHN F KENNEDY S/N Y AV MARISCAL SUCRE,LAIART CUENCA,1.0,0.800000,2.0,1.600000,3.200000


In [548]:
filtrada_score = base_registro_civil_test[(base_registro_civil_test['matcheo_qgram'].notna()) & (base_registro_civil_test['porcentaje_match_02']>=1)][['numero_establecimiento','numero_ruc','matcheo_qgram', 'score_qgram_02', 'nombre_fantasia_comercial','calles', 'porcentaje_match_02', 'score_general']]

In [549]:
len(filtrada_score)

193

In [550]:
len(recreo_nombres)

198

In [551]:
filtrada_score[filtrada_score['nombre_fantasia_comercial'].str.contains('campero', case = False, na = False)]

,numero_establecimiento,numero_ruc,matcheo_qgram,score_qgram_02,nombre_fantasia_comercial,calles,porcentaje_match_02,score_general
7222407,12.0,1.791925e+12,CAMPERO,0.909091,pollo campero,AVENIDA LA PRENSA S/N Y AVENIDA MARISCAL SUCRE,1.111111,1.010101
7231637,25.0,1.791998e+12,CAMPERO,0.909091,pollo campero,AV LA PRENSA 5425 Y AV DEL MAESTRO,1.111111,1.010101
7231632,20.0,1.791998e+12,CAMPERO,0.909091,pollo campero,AV. LA PRENSA S/N Y AV.MARISCAL SUCRE,1.555556,1.414141
7222398,3.0,1.791925e+12,CAMPERO,2.000000,campero,AV. DE LA PRENSA 5425 Y AV. DEL MAESTRO,1.000000,2.000000
7183526,4.0,1.791352e+12,CAMPERO,2.000000,campero,AV LA PRENSA 5425 Y AV DEL MAESTRO,1.111111,2.222222
7260980,2.0,1.792241e+12,CAMPERO,2.000000,campero,AV. MARISCAL SUCRE S/N Y JOHN F. KENNEDY,1.555556,3.111111


In [517]:
NO_COINCIDE = ['NO COINCIDE RUC', 'NO HAY EN CONDADO', 'NO HAY ESPECIFICO PARA CONDADO', 'NO HAY ESPECIFICO']

In [486]:
set(recreo_nombres[~recreo_nombres['CÓDIGO ESTABLECIMIENTO'].isin(NO_COINCIDE)]['nombre_establecimiento'])-set(filtrada_score['matcheo_qgram'])  

{'ACCESORIOS CELULARES',
 'ADRISSA',
 'ALDO',
 'ALMACENES BANDA',
 'AMERICAN CLASSICS',
 'AMERICAN DELI',
 'ANETA',
 'BANCO AUSTRO- y ATM B. AUSTRO',
 'BANCO BOLIVARIANO',
 'BANCO DE GUAYAQUIL',
 'BANCO DEL PACIFICO',
 'BANCO INTERNACIONAL',
 'BANCO PICHINCHA - LOCAL Y CAJERO',
 'BANCO SOLIDARIO',
 'BASKIN ROBBINS ',
 'BASSA',
 'BATA',
 'BATH & BODY',
 'BEBE MUNDO ',
 'BERSHKA',
 'BUFFALOS',
 'BURGUER KING',
 'CALL IT SPRING',
 'CALZADO BIBI',
 'CAMPERO',
 'CASA BRASIL',
 'CASABACA TOYOTA- TALLERES',
 'CHAIDE  y TEMPUR',
 'CHERY(maresa) ',
 'CHEVROLET PROAUTO - TALLERES',
 'CHICBERRY FROZEN YOGURT',
 'CLARO',
 'CLEANEXPRESS',
 'CNT LOCAL y ANTENAS',
 'CONVERSE',
 'COOK - INSUMOS',
 'CORPORACION EUGENIO ESPEJO',
 'CREPES & WAFFLES ',
 'CROCS',
 'CYRANO',
 'D BOND',
 'DELBANK',
 'DENTAL SI',
 'DONUT EXPRESS',
 'EL ESPAÑOL',
 'ES PRINT PLOTTER',
 'ETAFASHION',
 'EXPLORER',
 'FARMACIA NATURAL VITALITY',
 'FIVE STARS',
 'FUN RIDES',
 'FUNKY FISH',
 'FYBECA',
 'GAMES & GAMES',
 'GLOW',
 'GO 

In [528]:
df = base_registro_civil_test
df[(df['nombre_fantasia_comercial'].str.contains(r'(?=.*campero)', case = False, na = False)) & (df['canton'].str.contains('Quito', case = False, na = False)) ]

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,provincia,canton,parroquia,calles,matcheo_qgram,score_qgram,porcentaje_match,score_qgram_02,porcentaje_match_02,score_general
7231637,1.791998e+15,1.791998e+12,25.0,ENMARSI S.A.,pollo campero,1.0,ACTIVO,1.0,ABIERTO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV LA PRENSA 5425 Y AV DEL MAESTRO,NaN,0.454545,0.555556,0.909091,1.111111,1.010101
7222407,1.791925e+15,1.791925e+12,12.0,ADMINELI CIA. LTDA.,pollo campero,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AVENIDA LA PRENSA S/N Y AVENIDA MARISCAL SUCRE,NaN,0.454545,0.555556,0.909091,1.111111,1.010101
7231632,1.791998e+15,1.791998e+12,20.0,ENMARSI S.A.,pollo campero,1.0,ACTIVO,1.0,ABIERTO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV. LA PRENSA S/N Y AV.MARISCAL SUCRE,NaN,0.454545,0.777778,0.909091,1.555556,1.414141
7222398,1.791925e+15,1.791925e+12,3.0,ADMINELI CIA. LTDA.,campero,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV. DE LA PRENSA 5425 Y AV. DEL MAESTRO,CAMPERO,1.000000,0.500000,2.000000,1.000000,2.000000
7183526,1.791352e+15,1.791352e+12,4.0,ALIMENTOS PREPARADOS C.A. ALIPRECA,campero,2.0,PASIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV LA PRENSA 5425 Y AV DEL MAESTRO,CAMPERO,1.000000,0.555556,2.000000,1.111111,2.222222
7260980,1.792241e+15,1.792241e+12,2.0,ANIACOMERCIAL S.A.,campero,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV. MARISCAL SUCRE S/N Y JOHN F. KENNEDY,CAMPERO,1.000000,0.777778,2.000000,1.555556,3.111111


In [426]:
len(recreo_nombres)

198

In [383]:
percentiles = [i/10 for i in range(1,10)]

In [384]:
base_registro_civil_test[(base_registro_civil_test['matcheo_qgram'].notna()) & ~(base_registro_civil_test['nombre_fantasia_comercial'].str.contains('recreo', case= False, na = False))][['score_general', 'porcentaje_match_02', 'score_qgram_02']].describe(percentiles)

,score_general,porcentaje_match_02,score_qgram_02
count,39.000000,39.000000,39.000000
mean,2.616359,1.337490,1.951426
std,0.519665,0.242247,0.115552
min,1.500000,0.750000,1.652174
10%,1.687179,0.888889,1.692308
20%,2.319784,1.200000,2.000000
30%,2.518182,1.296970,2.000000
40%,2.666667,1.333333,2.000000
50%,2.666667,1.400000,2.000000
60%,2.800000,1.400000,2.000000


quedarnos con los que son explicitos de el recreo y luego definir un umbral

In [385]:
base_registro_civil_test[(base_registro_civil_test['matcheo_qgram'].notna())][['score_general', 'porcentaje_match_02', 'score_qgram_02']].describe(percentiles)

,score_general,porcentaje_match_02,score_qgram_02
count,39.000000,39.000000,39.000000
mean,2.616359,1.337490,1.951426
std,0.519665,0.242247,0.115552
min,1.500000,0.750000,1.652174
10%,1.687179,0.888889,1.692308
20%,2.319784,1.200000,2.000000
30%,2.518182,1.296970,2.000000
40%,2.666667,1.333333,2.000000
50%,2.666667,1.400000,2.000000
60%,2.800000,1.400000,2.000000
